# 🏆 Entrenamiento del Modelo v6 — más partidas de maestros, mejor validación

Entrenamiento en Google Colab con GPU (T4 o superior). **Qué cambia respecto de la v5** (y por qué):

| | v5 | v6 |
|---|---|---|
| Partidas | 12.000 | **decenas de miles** (se decide con el conteo de la sección 3) |
| Filtro de ELO | ambos ≥ 2000 | **ambos ≥ 2200** (configurable; se eligió con el conteo de la sección 3) |
| Validación | 90/10 mezclando **posiciones sueltas** | **por partida entera** (sin filtrar información entre train y val) |
| Posiciones en memoria | `float32` (3 KB c/u) | `uint8` (768 B c/u): cuatro veces menos |
| Evaluación | partidas de cualquier nivel | **partidas de maestros que el modelo no vio** |
| Entrenamiento | FP32 | precisión mixta (más rápido en T4) |
| Reanudación | no | **sí**: guarda cada época y retoma si Colab se desconecta |

> **Por qué importa la validación por partida:** en la v5 las posiciones de una misma partida podían quedar
> unas en entrenamiento y otras en validación, así que el 42,1 % de precisión que se reportó es optimista. La v6
> no repite ese error, por eso su número **no es comparable** con el de la v5: para comparar de verdad usá la
> sección 9, que evalúa las dos con las mismas partidas nuevas.

> **Regla 1 del proyecto:** este entrenamiento usa solo partidas humanas de Lichess. Stockfish aparece únicamente
> en la evaluación (como vara de comparación), nunca para decidir las jugadas del modelo.

> **Antes de empezar:** activá la GPU (`Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU`) y
> asegurate de haber hecho `git push` de los últimos cambios de `training/` (este notebook clona el repo de GitHub).

## 1. Clonar el repositorio

In [ ]:
import os
import sys

%cd /content
!rm -rf /content/ajedrez
!git clone --depth 1 https://github.com/larzekao9/Brazo-Rob-tico-con-Inteligencia-Artificial-para-el-Aprendizaje-del-Ajedrez.git /content/ajedrez
%cd /content/ajedrez

if '/content/ajedrez' not in sys.path:
    sys.path.insert(0, '/content/ajedrez')

# Las dependencias se instalan en la sección 2; recién después se importa el pipeline.
# Si más adelante falta `contar_partidas_por_elo`, hay que hacer `git push` de training/data_pipeline.py.
assert os.path.exists('training/data_pipeline.py'), 'No se clonó el repositorio'
print('✅ Repositorio clonado')

## 2. Dependencias y GPU
`python-chess` y `zstandard` con versión fija, y Stockfish solo para la evaluación (nunca decide jugadas).

In [ ]:
import torch

!pip install -q python-chess==1.999.0 zstandard==0.25.0
!apt-get update -qq && apt-get install -y -qq stockfish
!ln -sf /usr/games/stockfish /usr/local/bin/stockfish

dispositivo = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🚀 Dispositivo: {dispositivo}')
if dispositivo == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️ Sin GPU: activala en Entorno de ejecución > Cambiar tipo de entorno de ejecución.')

## 3. Dataset de Lichess y conteo de partidas por ELO
Descarga el mismo archivo que usó la v5 (febrero 2017, ~1,8 GB) y **cuenta cuántas partidas hay con ambos jugadores
arriba de cada ELO**. Con ese número se decide el filtro de la sección 4 sin adivinar.

> `LIMITE_CONTEO = None` cuenta el archivo entero (tarda más); con un número cuenta solo las primeras partidas
> y alcanza para estimar la proporción.

In [ ]:
!mkdir -p training/data
!wget -c -q --show-progress -O training/data/lichess_db_standard_rated_2017-02.pgn.zst \
    https://database.lichess.org/standard/lichess_db_standard_rated_2017-02.pgn.zst
!ls -lh training/data/

In [ ]:
import time

from training.data_pipeline import contar_partidas_por_elo

RUTA_DATASET = 'training/data/lichess_db_standard_rated_2017-02.pgn.zst'
LIMITE_CONTEO = 3_000_000   # None = todo el archivo

t0 = time.time()
conteo = contar_partidas_por_elo(RUTA_DATASET, umbrales=(2000, 2200, 2300, 2400, 2500), limite_leidas=LIMITE_CONTEO)
print(f"Partidas leídas: {conteo['leidas']:,} ({time.time() - t0:.0f}s) | con ELO válido: {conteo['con_elo']:,}")
for umbral, cantidad in conteo['por_umbral'].items():
    pct = 100 * cantidad / max(conteo['con_elo'], 1)
    print(f'  ambos jugadores ≥ {umbral}: {cantidad:>9,}  ({pct:.2f} %)')

## 4. Configuración del entrenamiento v6
Cambiá estos valores según el conteo anterior. **Memoria:** cada partida aporta ~79 posiciones de 768 bytes
(`uint8`); 35.000 partidas ≈ 2,8 millones de posiciones ≈ 2,1 GB. Si Colab se queda sin memoria, bajá `LIMITE_PARTIDAS`.

In [ ]:
import datetime
import os

ELO_MINIMO = 2200            # ambos jugadores (con 2300 hay pocas partidas: ~14.500 cada 3 millones)
LIMITE_PARTIDAS = 35_000     # partidas a usar (train + validación); se corta antes si el archivo no alcanza
FRACCION_VALIDACION = 0.03   # partidas enteras reservadas para validar
SEMILLA = 42

CANALES = 192                # igual que la v5, para que la mejora se deba a los datos y no al tamaño de la red
BLOQUES = 8                  # (probá 256 y 12 si querés una red más grande: pesa más y la inferencia tarda más)
EPOCAS = 30
TAMANO_LOTE = 256
LR_MAXIMO = 1e-3
USAR_DRIVE = True            # guarda cada época en Drive para poder retomar si Colab se desconecta

if USAR_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    CARPETA_CHECKPOINTS = '/content/drive/MyDrive/ajedrez_checkpoints/v6'
else:
    CARPETA_CHECKPOINTS = '/content/checkpoints/v6'
os.makedirs(CARPETA_CHECKPOINTS, exist_ok=True)
FECHA = datetime.date.today().isoformat()
print('📁 Checkpoints en:', CARPETA_CHECKPOINTS)

## 5. Extracción de partidas de maestros (validación separada por partida)
Lee el archivo en streaming y separa **partidas enteras** entre entrenamiento y validación. Guarda
`partidas_crudas_leidas`: hasta qué partida del archivo llegó, para que la evaluación use partidas posteriores.

In [ ]:
import numpy as np

from training.data_pipeline import pgn_a_muestras_divididas

t0 = time.time()
extraccion = pgn_a_muestras_divididas(
    RUTA_DATASET,
    limite_partidas=LIMITE_PARTIDAS,
    elo_minimo=ELO_MINIMO,
    fraccion_validacion=FRACCION_VALIDACION,
    semilla=SEMILLA,
)
print(f'✅ Extraído en {time.time() - t0:.0f}s')
print(f'   Partidas: train {extraccion.partidas_train:,} | validación {extraccion.partidas_val:,}')
print(f'   Posiciones: train {len(extraccion.muestras_train):,} | validación {len(extraccion.muestras_val):,}')
print(f'   Llegó hasta la partida {extraccion.partidas_crudas_leidas:,} del archivo')
PARTIDAS_CRUDAS_LEIDAS = extraccion.partidas_crudas_leidas

# Se pasa a arreglos compactos (uint8) y se libera la lista de tuplas.
X_train = np.stack([t for t, _ in extraccion.muestras_train])
y_train = np.array([e for _, e in extraccion.muestras_train], dtype=np.int64)
X_val = np.stack([t for t, _ in extraccion.muestras_val])
y_val = np.array([e for _, e in extraccion.muestras_val], dtype=np.int64)
CANTIDAD_MUESTRAS = len(y_train) + len(y_val)
PARTIDAS_USADAS = extraccion.partidas_train + extraccion.partidas_val
del extraccion
import gc; gc.collect()
print(f'   X_train: {X_train.shape} {X_train.dtype} = {X_train.nbytes / 1e9:.2f} GB')

## 6. Entrenamiento (SE-ResNet, precisión mixta, con reanudación)
Cada época guarda `v6_ultimo.pt` (pesos + optimizador + planificador). **Si Colab se desconecta, volvé a correr las
secciones 1 a 5 y esta: retoma desde la última época guardada.** Las posiciones viven en la GPU como `uint8` y se
pasan a `float32` por lote.

In [ ]:
import torch
from backend.servicios.aprendizaje.modelo_jugadas import NUM_CLASES, RedSEResNetAjedrez

torch.manual_seed(SEMILLA)
red = RedSEResNetAjedrez(canales=CANALES, cantidad_bloques=BLOQUES, cantidad_clases=NUM_CLASES).to(dispositivo)
print(f'Parámetros: {sum(p.numel() for p in red.parameters()):,}')

optimizador = torch.optim.AdamW(red.parameters(), lr=LR_MAXIMO, weight_decay=1e-4)
pasos_por_epoca = (len(y_train) + TAMANO_LOTE - 1) // TAMANO_LOTE
# 1 época de calentamiento lineal y luego coseno hasta el final.
calentamiento = torch.optim.lr_scheduler.LinearLR(optimizador, start_factor=0.1, total_iters=pasos_por_epoca)
coseno = torch.optim.lr_scheduler.CosineAnnealingLR(optimizador, T_max=(EPOCAS - 1) * pasos_por_epoca, eta_min=1e-5)
scheduler = torch.optim.lr_scheduler.SequentialLR(optimizador, [calentamiento, coseno], milestones=[pasos_por_epoca])
funcion_perdida = torch.nn.CrossEntropyLoss(label_smoothing=0.05)
escalador = torch.amp.GradScaler(enabled=(dispositivo == 'cuda'))

# Datos en la GPU (uint8) para no depender del DataLoader.
Xt = torch.from_numpy(X_train).to(dispositivo)
yt = torch.from_numpy(y_train).to(dispositivo)
Xv = torch.from_numpy(X_val).to(dispositivo)
yv = torch.from_numpy(y_val).to(dispositivo)

def a_entrada(lote_uint8):
    # (N, 8, 8, 12) uint8 -> (N, 12, 8, 8) float32, el mismo formato que `tensor_a_entrada_red`.
    return lote_uint8.permute(0, 3, 1, 2).float()

@torch.no_grad()
def evaluar():
    red.eval()
    top1 = top3 = 0
    for i in range(0, len(yv), 1024):
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=(dispositivo == 'cuda')):
            salida = red(a_entrada(Xv[i:i + 1024]))
        etiquetas = yv[i:i + 1024]
        top1 += (salida.argmax(1) == etiquetas).sum().item()
        top3 += (salida.topk(3, dim=1).indices == etiquetas.unsqueeze(1)).any(1).sum().item()
    return top1 / len(yv), top3 / len(yv)

RUTA_ULTIMO = f'{CARPETA_CHECKPOINTS}/v6_ultimo.pt'
RUTA_MEJOR = f'{CARPETA_CHECKPOINTS}/v6_mejor.pt'
epoca_inicial, mejor_acc = 1, -1.0   # -1: la primera época siempre guarda un "mejor" aunque la precisión sea 0
if os.path.exists(RUTA_ULTIMO):
    estado = torch.load(RUTA_ULTIMO, map_location=dispositivo)
    red.load_state_dict(estado['state_dict'])
    optimizador.load_state_dict(estado['optimizador'])
    scheduler.load_state_dict(estado['scheduler'])
    epoca_inicial, mejor_acc = estado['epoca'] + 1, estado['mejor_acc']
    print(f'♻️ Retomando desde la época {epoca_inicial} (mejor acc. hasta ahora: {mejor_acc:.2%})')

metadatos = dict(
    num_clases=NUM_CLASES, arquitectura='se_resnet', canales=CANALES, cantidad_bloques=BLOQUES,
    cantidad_partidas=PARTIDAS_USADAS, elo_minimo=ELO_MINIMO, cantidad_muestras=CANTIDAD_MUESTRAS,
    validacion='por_partida', fecha=FECHA,
)

for epoca in range(epoca_inicial, EPOCAS + 1):
    t_epoca = time.time()
    red.train()
    permutacion = torch.randperm(len(yt), device=dispositivo)
    perdida_acumulada = 0.0
    for i in range(0, len(yt), TAMANO_LOTE):
        indices = permutacion[i:i + TAMANO_LOTE]
        entradas, etiquetas = a_entrada(Xt[indices]), yt[indices]
        optimizador.zero_grad(set_to_none=True)
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=(dispositivo == 'cuda')):
            perdida = funcion_perdida(red(entradas), etiquetas)
        escalador.scale(perdida).backward()
        escalador.step(optimizador)
        escalador.update()
        scheduler.step()
        perdida_acumulada += perdida.item() * len(indices)

    acc1, acc3 = evaluar()
    print(f'Época {epoca:02d}/{EPOCAS} | LR {optimizador.param_groups[0]["lr"]:.6f} | '
          f'Pérdida {perdida_acumulada / len(yt):.4f} | Val top-1 {acc1:.2%} | top-3 {acc3:.2%} | {time.time() - t_epoca:.0f}s')

    if acc1 > mejor_acc:
        mejor_acc = acc1
        torch.save({'state_dict': red.state_dict(), **metadatos, 'epoca': epoca, 'accuracy_val': acc1, 'accuracy_val_top3': acc3}, RUTA_MEJOR)
        print(f'  ⭐ Nuevo mejor ({acc1:.2%})')
    torch.save({'state_dict': red.state_dict(), 'optimizador': optimizador.state_dict(),
                'scheduler': scheduler.state_dict(), 'epoca': epoca, 'mejor_acc': mejor_acc}, RUTA_ULTIMO)

print(f'\n🏆 Entrenamiento terminado. Mejor precisión de validación (por partida): {mejor_acc:.2%}')

## 7. Exportar el checkpoint v6
Guarda el mejor modelo con todos sus metadatos (`modelo_jugadas_v6_<fecha>.pt`) y lo descarga a tu computadora.
**No se sube al repositorio** (los checkpoints están en `.gitignore`): se copia a `training/checkpoints/` y a Drive.

In [ ]:
from google.colab import files

mejor = torch.load(RUTA_MEJOR, map_location='cpu')
nombre_checkpoint = f'modelo_jugadas_v6_{FECHA}.pt'
ruta_checkpoint = f'{CARPETA_CHECKPOINTS}/{nombre_checkpoint}'
torch.save({k: v for k, v in mejor.items()}, ruta_checkpoint)
print(f"✅ Guardado: {ruta_checkpoint} | época {mejor['epoca']} | val top-1 {mejor['accuracy_val']:.2%}")
files.download(ruta_checkpoint)

## 8. Subir la v5 para compararla
Para comparar justo hace falta la v5 en esta sesión: subí `modelo_jugadas_v5_2026-09-24.pt` desde
`training/checkpoints/` de tu PC (o copialo a Drive y poné la ruta en `RUTA_V5`).

In [ ]:
from google.colab import files

RUTA_V5 = '/content/modelo_jugadas_v5_2026-09-24.pt'
if not os.path.exists(RUTA_V5):
    subidos = files.upload()
    nombre = next(iter(subidos))
    os.replace(nombre, RUTA_V5)
print('✅ v5 disponible:', RUTA_V5)

## 9. Evaluación comparativa v5 vs v6 (con partidas que ninguna vio)
Las dos versiones se evalúan con **las mismas partidas de maestros posteriores a donde llegó la extracción**
(`PARTIDAS_CRUDAS_LEIDAS` + margen), así ninguna las vio en entrenamiento. Mide la coincidencia con la jugada del
maestro y qué tan cara sale cada desacierto según Stockfish (que aquí **solo mide**, no decide).

> Cada desacierto llama a Stockfish, por eso `CANTIDAD_PARTIDAS_EVAL` es chico. Subilo si querés más precisión
> estadística (30 partidas son ~2.000 jugadas: el resultado es orientativo, no concluyente).

In [ ]:
from training.evaluar_modelo import evaluar_modelo, BALDES_ERROR

CANTIDAD_PARTIDAS_EVAL = 30
SALTAR_EVAL = PARTIDAS_CRUDAS_LEIDAS + 1000

resultados = {}
for etiqueta, ruta in (('v5', RUTA_V5), ('v6', ruta_checkpoint)):
    print(f'--- Evaluando {etiqueta} ---')
    resultados[etiqueta] = evaluar_modelo(
        RUTA_DATASET,
        cantidad_partidas=CANTIDAD_PARTIDAS_EVAL,
        saltar_partidas=SALTAR_EVAL,
        ruta_checkpoint=ruta,
        elo_minimo=ELO_MINIMO,
    )

print()
print(f"{'':14s} {'v5':>10s} {'v6':>10s}")
print(f"{'Jugadas':14s} {resultados['v5'].total_jugadas:>10,d} {resultados['v6'].total_jugadas:>10,d}")
print(f"{'Top-1 (coincide con el maestro)':14s} {resultados['v5'].accuracy:>10.2%} {resultados['v6'].accuracy:>10.2%}")
for balde in BALDES_ERROR:
    p5 = resultados['v5'].distribucion_errores[balde] / resultados['v5'].total_jugadas
    p6 = resultados['v6'].distribucion_errores[balde] / resultados['v6'].total_jugadas
    print(f'{balde.capitalize():14s} {p5:>10.1%} {p6:>10.1%}')

## 10. ¿Se puede subir el tope de nivel del modelo (hoy 18)?
**No se decide con esta tabla.** `NIVEL_MAX_MODELO` está en 18 porque el modelo se entrenó con ELO ≥ 2000. Para subirlo
hay que **medir partidas completas** de la v6 contra Stockfish en niveles 19 y 20 y ver si las gana o empata de forma
sostenida. Esa evaluación (modelo contra motor, partida entera) es el siguiente paso de HU4: si la v6 no la supera,
el tope se queda en 18 aunque la precisión mejore.

## 11. Demostración de inferencia con la v6

In [ ]:
from backend.servicios.aprendizaje.inferencia import predecir_jugada_maestra, predecir_top_jugadas

fen_test = "r1bqkb1r/pppp1ppp/2n5/4p3/2B1n3/5N2/PPPP1PPP/RNBQK2R w KQkq - 0 5"
t0 = time.perf_counter()
jugada = predecir_jugada_maestra(fen_test, ruta_checkpoint=ruta_checkpoint)
print(f'Jugada: {jugada} | latencia {(time.perf_counter() - t0) * 1000:.1f} ms')
for j, prob in predecir_top_jugadas(fen_test, top_n=3, ruta_checkpoint=ruta_checkpoint):
    print(f'  • {j:6s} {prob * 100:.2f}%')